<a href="https://colab.research.google.com/github/vituhaa/recsys_projects/blob/main/user2user.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Скачиваем датасет
path = kagglehub.dataset_download("trishna8/movielens-100k-dataset")
print("Путь:", path)

Using Colab cache for faster access to the 'movielens-100k-dataset' dataset.
Путь: /kaggle/input/movielens-100k-dataset


In [2]:
import os

for file in os.listdir(path):
    print(f"  {file}")

  ml-100k


In [3]:
import os
import pandas as pd

ds_path = '/kaggle/input/movielens-100k-dataset/ml-100k'

ratings_file = os.path.join(ds_path, "u.data")

ratings = pd.read_csv(
    ratings_file,
    sep="\t",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies_file = os.path.join(ds_path, "u.item")

movies = pd.read_csv(
    movies_file,
    sep="|",
    names=["movieId", "title"],
    encoding="latin-1"
)

# сортируем по пользователю и времени
ratings_numbered = (
    ratings
    .sort_values(["userId", "timestamp"])
    .copy()
)

# номер взаимодействия внутри пользователя
ratings_numbered["row_num"] = (
    ratings_numbered
    .groupby("userId")
    .cumcount() + 1
)

# количество оценок у пользователя
ratings_numbered["user_count"] = (
    ratings_numbered
    .groupby("userId")["movieId"]
    .transform("count")
)

# точка разделения 80/20
ratings_numbered["split_point"] = (
    ratings_numbered["user_count"] * 0.8
).astype(int)


# train
train = (
    ratings_numbered[
        ratings_numbered["row_num"] <= ratings_numbered["split_point"]
    ]
    .drop(columns=["row_num", "user_count", "split_point"])
)


# test
test = (
    ratings_numbered[
        ratings_numbered["row_num"] > ratings_numbered["split_point"]
    ]
    .drop(columns=["row_num", "user_count", "split_point"])
)


print("Train:", len(train))
print("Test:", len(test))

Train: 79619
Test: 20381


In [4]:
import pandas as pd

pd_train = train[["userId", "movieId", "rating"]]

user_item = pd_train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating",
    fill_value=0
)

user_item.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,1662,1663,1664,1670,1672,1673,1675,1676,1679,1681
userId,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,0.0,3.0,0.0,0.0,4.0,1.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
def cosine_users(user1, user2, user_item):
    v1 = user_item.loc[user1]
    v2 = user_item.loc[user2]

    dot = v1.dot(v2)

    norm1 = (v1 ** 2).sum() ** 0.5
    norm2 = (v2 ** 2).sum() ** 0.5

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot / (norm1 * norm2)

In [6]:
def find_similar_users(user_id, user_item, top_n):
    similarities = []

    for other_user in user_item.index:
        if other_user == user_id:
            continue

        similarity = cosine_users(
            user_id,
            other_user,
            user_item
        )

        similarities.append(
            (other_user, similarity)
        )

    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return similarities[:top_n]

In [8]:
def recommend_user_user(user_id, user_item, top_k=10, n_neighbors=20):
    neighbors = find_similar_users(
        user_id,
        user_item,
        top_n=n_neighbors
    )

    user_ratings = user_item.loc[user_id]

    watched_movies = set(
        user_ratings[user_ratings > 0].index # берём номера фильмов
    )

    scores = {}
    sim_sum = {}

    for neighbor_id, similarity in neighbors:
        ratings = user_item.loc[neighbor_id]
        for movie_id in user_item.columns:
            rating = ratings[movie_id]

            if rating == 0 or movie_id in watched_movies:
                continue

            scores[movie_id] = (
                scores.get(movie_id, 0)
                + similarity * rating
            )
            sim_sum[movie_id] = (
                sim_sum.get(movie_id, 0)
                + similarity
            )

    recommendations = sorted(
        (
          (
              movie_id,
              scores[movie_id] / sim_sum[movie_id]
          )
          for movie_id in scores
        ),
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations[:top_k]

In [9]:
recommend_user_user(
    user_id=1,
    user_item=user_item,
    top_k=10
)

[(331, np.float64(5.0)),
 (1268, np.float64(5.0)),
 (1009, np.float64(5.0)),
 (285, np.float64(5.0)),
 (1589, np.float64(5.0)),
 (492, np.float64(5.0)),
 (602, np.float64(5.0)),
 (736, np.float64(5.0)),
 (1007, np.float64(5.0)),
 (1143, np.float64(5.0))]

In [10]:
pd_test = test[["userId", "movieId", "rating"]]

In [11]:
pd_test.head()

,userId,movieId,rating
8976,1,12,5
9811,1,201,3
10508,1,208,5
74847,1,116,3
78171,1,58,4


In [16]:
import math

def evaluate_user2user(pd_test, user_item, k=10, max_users=20381):
    precision_list = []
    recall_list = []
    ap_list = []

    users = pd_test["userId"].unique()

    if max_users:
        users = users[:max_users]

    for user_id in users:
        true_movies = set(
            pd_test[
                (pd_test["userId"] == user_id) &
                (pd_test["rating"] >= 4)
            ]["movieId"]
        )

        if len(true_movies) == 0:
            continue

        recommendations = recommend_user_user(
            user_id=user_id,
            user_item=user_item,
            top_k=k
        )

        rec_movies = [movie_id for movie_id, score in recommendations]

        hits = [1 if movie_id in true_movies else 0
            for movie_id in rec_movies]

        n_hits = sum(hits)

        precision = n_hits / k
        recall = n_hits / len(true_movies)

        ap_sum = 0

        for rank, hit in enumerate(hits, start=1):
            if hit:
                precision_at_rank = sum(hits[:rank]) / rank
                ap_sum += precision_at_rank

        ap = ap_sum / min(len(true_movies), k)

        precision_list.append(precision)
        recall_list.append(recall)
        ap_list.append(ap)

    return {
        f"Precision@{k}": sum(precision_list) / len(precision_list),
        f"Recall@{k}": sum(recall_list) / len(recall_list),
        f"MAP@{k}": sum(ap_list) / len(ap_list),
    }

In [17]:
metrics = evaluate_user2user(
    pd_test,
    user_item,
    k=10,
    max_users=30
)

metrics

{'Precision@10': 0.013333333333333334,
 'Recall@10': 0.0055272108843537416,
 'MAP@10': 0.0020833333333333333}

In [39]:
recommend_user_user(
    user_id=300,
    user_item=user_item,
    top_k=30
)

[(313, np.float64(5.0)),
 (902, np.float64(5.0)),
 (347, np.float64(5.0)),
 (340, np.float64(5.0)),
 (892, np.float64(5.0)),
 (1, np.float64(5.0)),
 (117, np.float64(5.0)),
 (591, np.float64(5.0)),
 (742, np.float64(5.0)),
 (887, np.float64(4.999999999999999)),
 (984, np.float64(4.691985260708939)),
 (237, np.float64(4.476521005789735)),
 (292, np.float64(4.215568948619129)),
 (301, np.float64(4.165168389320724)),
 (342, np.float64(4.081606783713652)),
 (339, np.float64(4.069413853569693)),
 (271, np.float64(4.00130946199163)),
 (1025, np.float64(4.0000560656681685)),
 (345, np.float64(4.0)),
 (750, np.float64(4.0)),
 (1527, np.float64(4.0)),
 (270, np.float64(4.0)),
 (329, np.float64(4.0)),
 (874, np.float64(4.0)),
 (886, np.float64(4.0)),
 (937, np.float64(4.0)),
 (1265, np.float64(4.0)),
 (751, np.float64(4.0)),
 (242, np.float64(4.0)),
 (311, np.float64(4.0))]